# AETHER STT — phase 1 (CTC branch) training notebook

Clones `aether-v3` from GitHub and runs the CTC-only pipeline: frozen Mimi encoder → semantic codes → `AetherSpeech` transformer → CTC head, on LibriSpeech.

The dummy-dataset smoke test (pipeline sanity check on a handful of utterances) already ran and passed locally - no need to repeat it here. This notebook goes straight to the real LibriSpeech run.

**Cross-session caching:** the ~30GB raw LibriSpeech download is disposable, but the *extracted* Mimi cache it produces (semantic codes + byte targets) is only ~100-150MB. This notebook mirrors that small extracted cache — and training checkpoints — to Google Drive, so a fresh Colab session (or a runtime-type switch) never has to redownload or re-extract from scratch, and a dropped session resumes from its last checkpoint instead of losing the compute units already spent.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/karl4th/aether-v3.git"
REPO_NAME = "aether-v3"

# Idempotent regardless of how many times this cell is re-run in the same
# kernel session: after the first run cwd is already inside the repo (from
# os.chdir below), so checking os.path.isdir("aether-v3") relative to cwd
# would look one level too deep and clone a second copy nested inside the
# first - repeatable indefinitely. Instead, explicitly handle "already
# standing inside the repo" as its own case.
cwd = Path.cwd()
if cwd.name == REPO_NAME and (cwd / ".git").is_dir():
    subprocess.run(["git", "pull"], check=True, cwd=cwd)
    repo_dir = cwd
    print("Already inside the repo, pulled.")
else:
    repo_dir = cwd / REPO_NAME
    if (repo_dir / ".git").is_dir():
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)
        print("Pulled")
    elif repo_dir.exists():
        raise RuntimeError(
            f"{repo_dir} exists but isn't a git checkout (no .git/) - "
            "remove or rename it manually before re-running this cell."
        )
    else:
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
        print("Cloned")
    os.chdir(repo_dir)

print("cwd:", os.getcwd())

In [ ]:
import importlib.util
import subprocess
import sys


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


# Most GPU notebook images already ship a CUDA-matched torch build — don't
# clobber it. Only install if genuinely missing.
if importlib.util.find_spec("torch") is None:
    pip_install("torch")
if importlib.util.find_spec("torchaudio") is None:
    pip_install("torchaudio")

pip_install(
    "transformers>=5.17",  # <5.17 lacks MimiModel.get_audio_codes_mask - see mimi_wrapper.py
    "datasets>=2.19,<4.0",  # >=4.0 requires torchcodec + system ffmpeg for Audio decoding
    "soundfile",
    "librosa",  # datasets<4.0's Audio decode path needs this alongside soundfile
    "jiwer",
    "pyyaml",
    "numpy",
    "tqdm",
)

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
print("using device:", DEVICE)
assert DEVICE == "cuda", "No GPU visible on this VM - check the Colab session's accelerator."

## Google Drive cache

Mounts Drive and points training's checkpoints/logs there directly (survives a dropped/recycled session). The extraction cache itself stays on local disk for speed — the cell below mirrors only the small extracted result to/from Drive around the `prepare_cache` call, not the raw audio.

**GPU tier:** Mimi is a small model — extraction doesn't benefit from a strong GPU. If the cache isn't on Drive yet, a **T4 runtime is enough and burns far fewer compute units** for this step. Switch to a stronger GPU (A100/L4) only once the cache is confirmed pushed to Drive and you're about to train — reconnecting with a different runtime type resets local disk, but the next cell then restores the cache from Drive in seconds instead of re-extracting.

In [ ]:
import os
from pathlib import Path

from aether_v3.config import load_config

real_config = load_config("configs/ctc_base.yaml")

try:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/aether-v3")
except ImportError:
    DRIVE_ROOT = Path("drive_cache").resolve()
    print(f"Not running in Colab - falling back to local '{DRIVE_ROOT}' (no cross-session persistence).")

DRIVE_CACHE_DIR = DRIVE_ROOT / "data_cache" / Path(real_config.data.cache_dir).name
DRIVE_RUN_DIR = DRIVE_ROOT / "runs" / Path(real_config.train.output_dir).name

# Checkpoints/logs go straight to Drive - a dropped session shouldn't cost
# the training progress already paid for in GPU units.
real_config.train.output_dir = str(DRIVE_RUN_DIR)

# Match extraction parallelism to whatever this VM actually has.
real_config.data.extraction_num_workers = max(2, min(os.cpu_count() or 4, 16))

# Resume from a previous session's progress instead of starting over.
resume_path = DRIVE_RUN_DIR / "last.pt"
if resume_path.exists():
    real_config.train.resume_from = str(resume_path)
    print(f"Found existing checkpoint at {resume_path}, will resume from it.")

print("Drive cache dir:", DRIVE_CACHE_DIR)
print("Drive run dir:  ", DRIVE_RUN_DIR)

## Real run: LibriSpeech clean-100 + clean-360

**This downloads the full configured splits, not a sample** — unless the extracted cache is already on Drive (see above), in which case this cell just restores it and skips download+extraction entirely. On a cold cache: per `configs/ctc_base.yaml`, `train.100` (~6GB) + `train.360` (~24GB) for training, plus `dev-clean`/`dev-other`/`test-clean`/`test-other` (a few hundred MB each) for eval — roughly **30-35GB of raw audio**, decoded through Mimi and discarded; only the ~100-150MB extraction result is kept (locally, and mirrored to Drive below).

If you want to sanity-check on less data/compute before committing to the full 460h, edit `configs/ctc_base.yaml`'s `data.train_splits` down to just `["clean/train.100"]` before running this cell (and delete both the local `data_cache/ctc_base/` and its Drive mirror if you already ran it with the larger split — a mismatched fingerprint makes `prepare_cache` raise rather than silently reuse it).

Meant to run unattended on a rented GPU for a long time afterward - check `train.max_steps` / `train.batch_size` in the config for your hardware before launching training below.

In [ ]:
from aether_v3.data.cache_sync import hydrate_from_remote, push_to_remote
from aether_v3.data.mimi_cache import prepare_cache

restored = hydrate_from_remote(real_config.data.cache_dir, DRIVE_CACHE_DIR)
if restored:
    print(f"Restored from Drive, skipping re-extraction for: {restored}")

prepare_cache(real_config, device=DEVICE)

pushed = push_to_remote(real_config.data.cache_dir, DRIVE_CACHE_DIR)
if pushed:
    print(f"Pushed newly extracted cache to Drive: {pushed}")

### Train

**Single GPU:** run the Python cell below.

**Multiple GPUs on this machine:** don't use the Python cell — use the shell cell instead (`torchrun` spawns its own processes; the training loop auto-detects its environment variables and switches to DDP with no code changes).

Checkpoints/logs write straight to Drive (`real_config.train.output_dir`, set above), and if a `last.pt` was already there from a previous session, training resumes from it automatically — no edits needed after a dropped session or a runtime-type switch.

In [ ]:
from aether_v3.training.train_ctc import run_training

run_training(real_config)

In [ ]:
# Multi-GPU alternative to the cell above — edit nproc_per_node, then run this
# cell instead of the plain `run_training(real_config)` call.
# NPROC = 4
# !torchrun --nproc_per_node={NPROC} -m aether_v3.training.train_ctc --config configs/ctc_base.yaml

## Monitor training

Re-run this cell any time (even from a second notebook while the cell above is still training) to see the latest loss/WER/CER curves from `log.jsonl`.

Watch WER/CER, not eval loss - CTC-infeasible examples (target longer than the Mimi frame count) get `zero_infinity`-clamped to ~0 loss regardless of how well the model is actually doing, so eval loss alone is misleading.

In [ ]:
import json

import matplotlib.pyplot as plt

with open(f"{real_config.train.output_dir}/log.jsonl") as f:
    rows = [json.loads(line) for line in f]

train_rows = [r for r in rows if "loss" in r and "eval_loss" not in r]
eval_rows = [r for r in rows if "eval_cer" in r]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([r["step"] for r in train_rows], [r["loss"] for r in train_rows])
axes[0].set_title("train loss")
axes[0].set_xlabel("step")

axes[1].plot([r["step"] for r in eval_rows], [r["eval_cer"] for r in eval_rows], label="CER")
axes[1].plot([r["step"] for r in eval_rows], [r["eval_wer"] for r in eval_rows], label="WER")
axes[1].set_title("dev CER / WER")
axes[1].set_xlabel("step")
axes[1].legend()
plt.show()

if eval_rows:
    best = min(eval_rows, key=lambda r: r["eval_cer"])
    print("best eval so far:", best)